# OAI-ZIB (OAIZIB-CM) → nnU-Net v2 (từ HuggingFace)

Đơn giản nhất trong 3 dataset: HF `YongchengYAO/OAIZIB-CM` **đã sẵn định dạng nnU-Net** và nhãn **đã khớp bộ UNION** → chỉ cần **download → giải nén → dataset.json → verify** (không rasterize / remap / ghép ảnh).

- 404 train + 103 test. Nhãn 1–5 = femoral_bone, femoral_cartilage, tibial_bone, **medial_tibial_cartilage**, **lateral_tibial_cartilage**.
- OAI-ZIB là **nguồn DUY NHẤT có xương (1,3)** → khi merge, union 0–8 thành liên tiếp.
- Giữ `imagesTs` (103) làm **held-out bone+cartilage** cho bước đánh giá sau.


In [ ]:
!pip install -q huggingface_hub nibabel matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Cấu hình


In [ ]:
from pathlib import Path

DATASET_NAME = "Dataset001_KneeOA"
OUT_ROOT  = Path("/content/drive/MyDrive/nnUNet_raw") / DATASET_NAME   # cung cho Dataset011/012
IMAGES_TR = OUT_ROOT / "imagesTr"
LABELS_TR = OUT_ROOT / "labelsTr"
IMAGES_TS = OUT_ROOT / "imagesTs"
LABELS_TS = OUT_ROOT / "labelsTs"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Nhan UNION (khop SKM-TEA/iMorphics). OAI-ZIB label 4/5 chinh la med/lat tibial cartilage.
UNION_LABELS = {
    "background": 0, "femoral_bone": 1, "femoral_cartilage": 2,
    "tibial_bone": 3, "medial_tibial_cartilage": 4, "lateral_tibial_cartilage": 5,
}
print("OUT_ROOT =", OUT_ROOT)


## 1) Download từ HuggingFace (thẳng vào Drive)

Dataset **public** → không cần token. (Nếu bị gated: dùng Colab Secret rồi bỏ comment 2 dòng `login`.)


In [ ]:
from huggingface_hub import snapshot_download
# from google.colab import userdata; from huggingface_hub import login
# login(userdata.get("HF_TOKEN"))     # chi khi dataset gated; KHONG hardcode token

snapshot_download(repo_id="YongchengYAO/OAIZIB-CM", repo_type="dataset",
                  local_dir=str(OUT_ROOT))
print("Da tai ve:", OUT_ROOT)
print("Noi dung:", sorted(p.name for p in OUT_ROOT.iterdir()))


## 2) Giải nén các archive `.nii.gz` (dùng `zipfile`)

HF ship dạng zip; giải nén ra `imagesTr/oaizib_XXX_0000.nii.gz`, `labelsTr/oaizib_XXX.nii.gz` (và Ts) — **đã đúng tên nnU-Net**, khỏi đổi.


In [ ]:
import zipfile
for zn in ["imagesTr.zip", "labelsTr.zip", "imagesTs.zip", "labelsTs.zip", "info.zip"]:
    zp = OUT_ROOT / zn
    if zp.exists():
        with zipfile.ZipFile(zp) as z:
            z.extractall(OUT_ROOT)
        print("Giai nen:", zn)
    else:
        print("Bo qua (khong co):", zn)


## 3) Ghi `dataset.json`

`numTraining` đếm tự động từ `imagesTr`. Nhãn dùng tên UNION (med/lat tibial cartilage).


In [ ]:
import json, glob
n_train = len(glob.glob(str(IMAGES_TR / "*_0000.nii.gz")))
dataset = {
    "channel_names": {"0": "MRI"},
    "labels": UNION_LABELS,
    "numTraining": n_train,
    "file_ending": ".nii.gz",
    "name": "OAI_ZIB_KneeMRI",
    "description": "OAI-ZIB (OAIZIB-CM): bones + cartilage; nhan theo bo UNION.",
}
with open(OUT_ROOT / "dataset.json", "w") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print(json.dumps(dataset, indent=2, ensure_ascii=False))


## 4) Verify — đếm + nhãn + QC overlay


In [ ]:
import numpy as np, nibabel as nib

for name, d in [("imagesTr",IMAGES_TR),("labelsTr",LABELS_TR),("imagesTs",IMAGES_TS),("labelsTs",LABELS_TS)]:
    print(f"{name:10s}: {len(list(d.glob('*.nii.gz')))}")   # ky vong 404/404/103/103

print("\nUnique labels (5 ca dau labelsTr):")
for f in sorted(LABELS_TR.glob("*.nii.gz"))[:5]:
    u = np.unique(np.asanyarray(nib.load(str(f)).dataobj).astype(int))
    print(f"  {f.name}: {u.tolist()}")   # ky vong tap con cua {0,1,2,3,4,5}


In [ ]:
import matplotlib.pyplot as plt

def best_slice(lab):
    cands = []
    for ax in range(3):
        other = tuple(i for i in range(3) if i != ax)
        counts = (lab > 0).sum(axis=other)
        cands.append((int(counts.max()), ax, int(counts.argmax())))
    cands.sort(reverse=True)
    return cands[0][1], cands[0][2]

cid = sorted(LABELS_TR.glob("*.nii.gz"))[0].name.replace(".nii.gz", "")
img = np.asanyarray(nib.load(str(IMAGES_TR / f"{cid}_0000.nii.gz")).dataobj).astype(float)
lab = np.asanyarray(nib.load(str(LABELS_TR / f"{cid}.nii.gz")).dataobj).astype(int)
ax, z = best_slice(lab)
ims = np.take(img, z, axis=ax); lbs = np.take(lab, z, axis=ax)
imn = (ims - ims.min()) / (np.ptp(ims) + 1e-6)
plt.figure(figsize=(6, 6))
plt.imshow(imn.T, cmap="gray", origin="lower")
mm = np.ma.masked_where(lbs == 0, lbs)
plt.imshow(mm.T, cmap="nipy_spectral", alpha=0.5, origin="lower", vmin=1, vmax=5)
plt.title(f"{cid} (axis {ax}, slice {z}) - bone+cartilage"); plt.axis("off"); plt.show()
